<a href="https://colab.research.google.com/github/ANKIT-KANDULNA/CS318_DL-LAB/blob/main/Experiment-3/DL_lab_exp3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing the libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

Data Preprocessing

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

Dataset and Dataloader

In [4]:
train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=64,shuffle=False)

100%|██████████| 170M/170M [00:16<00:00, 10.1MB/s]


Activation Selector

In [5]:
def get_activation(name):
    if name == "relu":
        return nn.ReLU()
    elif name == "tanh":
        return nn.Tanh()
    elif name == "leaky_relu":
        return nn.LeakyReLU(0.01)


**CNN Class**

In [6]:
class SimpleCNN(nn.Module):
    def __init__(self,activation):
        super(SimpleCNN,self).__init__()

        self.act = get_activation(activation)

        self.conv1 = nn.Conv2d(3,32,kernel_size=3,padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        self.conv2 = nn.Conv2d(32,64,kernel_size=3,padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        self.pool = nn.MaxPool2d(2,2)
        self.dropout = nn.Dropout(0.25)

        self.fc1 = nn.Linear(64*8*8,128)
        self.fc2 = nn.Linear(128,10)

    def forward(self,x):
        x = self.pool(self.act(self.bn1(self.conv1(x))))
        x = self.pool(self.act(self.bn2(self.conv2(x))))
        x = x.view(x.size(0),-1)
        x = self.dropout(self.act(self.fc1(x)))
        x = self.fc2(x)
        return x


Weight Initialization

In [7]:
def init_weights(model,init_type):
    for m in model.modules():
        if isinstance(m,nn.Conv2d) or isinstance(m,nn.Linear):
            if init_type == "xavier":
                nn.init.xavier_uniform_(m.weight)
            elif init_type == "kaiming":
                nn.init.kaiming_uniform_(m.weight,nonlinearity="relu")
            elif init_type == "random":
                nn.init.normal_(m.weight,mean=0,std=0.01)


Optimizer Selection

In [8]:
def get_optimizer(name,model):
    if name == "sgd":
        return optim.SGD(model.parameters(),lr=0.01,momentum=0.9)
    elif name == "adam":
        return optim.Adam(model.parameters(),lr=0.001)
    elif name == "rmsprop":
        return optim.RMSprop(model.parameters(),lr=0.001)


Training Function

In [9]:
def train_model(model,optimizer,criterion,epochs=10):
    model.train()
    for epoch in range(epochs):
        running_loss = 0
        for images,labels in train_loader:
            images,labels = images.to(device),labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs,labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")


Evaluation Function

In [10]:
def evaluate_model(model):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images,labels in test_loader:
            images,labels = images.to(device),labels.to(device)
            outputs = model(images)
            _,predicted = torch.max(outputs,1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    return accuracy


In [12]:
activations = ["relu","tanh","leaky_relu"]
initializations = ["xavier","kaiming","random"]
optimizers = ["sgd","adam","rmsprop"]

criterion = nn.CrossEntropyLoss()

best_acc = 0
best_config = None

for act in activations:
    for init in initializations:
        for opt in optimizers:
            print(f"\nActivation:{act}, Init:{init}, Optimizer:{opt}")

            model = SimpleCNN(act).to(device)
            init_weights(model,init)

            optimizer = get_optimizer(opt,model)

            train_model(model,optimizer,criterion,epochs=2)
            acc = evaluate_model(model)

            print("Accuracy:",acc)

            if acc > best_acc:
                best_acc = acc
                best_config = (act,init,opt)
                torch.save(model.state_dict(),"best_cnn_cifar10.pth")



Activation:relu, Init:xavier, Optimizer:sgd
Epoch 1, Loss: 1.5735
Epoch 2, Loss: 1.2601
Accuracy: 61.7

Activation:relu, Init:xavier, Optimizer:adam
Epoch 1, Loss: 1.5693
Epoch 2, Loss: 1.2555
Accuracy: 62.84

Activation:relu, Init:xavier, Optimizer:rmsprop
Epoch 1, Loss: 1.9496
Epoch 2, Loss: 1.3867
Accuracy: 52.42

Activation:relu, Init:kaiming, Optimizer:sgd
Epoch 1, Loss: 1.6184
Epoch 2, Loss: 1.3228
Accuracy: 61.57

Activation:relu, Init:kaiming, Optimizer:adam
Epoch 1, Loss: 1.6183
Epoch 2, Loss: 1.3110
Accuracy: 63.38

Activation:relu, Init:kaiming, Optimizer:rmsprop
Epoch 1, Loss: 1.9483
Epoch 2, Loss: 1.3486
Accuracy: 58.7

Activation:relu, Init:random, Optimizer:sgd
Epoch 1, Loss: 1.4571
Epoch 2, Loss: 1.1127
Accuracy: 66.88

Activation:relu, Init:random, Optimizer:adam
Epoch 1, Loss: 1.3976
Epoch 2, Loss: 1.0717
Accuracy: 65.78

Activation:relu, Init:random, Optimizer:rmsprop
Epoch 1, Loss: 1.5703
Epoch 2, Loss: 1.2425
Accuracy: 61.62

Activation:tanh, Init:xavier, Optimize